# Module 7: Finding Failure Modes

> Self-directed module, ~30 min. UI-heavy — no code cells in this notebook.

By Module 6, the client research agent is deployed and the Studio testing in §8 has seeded a trace corpus in the LangSmith project. This module covers how to turn that raw trace data into signal that drives agent improvements — without manually reading every trace.


## The shift in scale

Manually reviewing traces was a viable workflow when agents were short and few. Three trends have pushed past that point:

- **Agents are longer-running.** A single trace can span dozens of tool calls and multiple subagent invocations. Even a careful reviewer needs minutes per trace; scanning hundreds is impractical.
- **Multimodal inputs and outputs are becoming common.** Traces include images, audio, attachments, and structured documents alongside text. Skimming for a pattern means inspecting more than text spans.
- **Agents are proliferating.** A typical enterprise deployment runs many agents, each producing its own trace stream. Even with focused attention, surface area outpaces a single reviewer.

LangSmith ships three features built around this problem. Each turns a different slice of trace data into action:

- **Polly** — a workspace AI assistant. Ask natural-language questions, get answers grounded in your project's traces, threads, prompts, and experiments.
- **Insights Agent** — an automated job that examines a trace corpus and produces hierarchical category reports. Surfaces patterns nobody knew to look for.
- **Engine** — a closed-loop system that detects recurring failures, diagnoses root causes, and proposes fixes as pull requests. Cloud-only at present.


---

## Polly

Polly is an AI assistant embedded directly in the LangSmith workspace. It lives in the bottom-right corner of every observability, prompt engineering, and evaluation page and answers natural-language questions about whatever you're currently looking at — traces, threads, prompts, experiments, datasets, queues.

Polly accesses the same data the UI does. The value isn't new visibility — it's skipping the steps of writing a filter expression, scrolling a long list, or opening several traces to compare them. For ad-hoc questions during development or review, it's the fastest path.

<img src="../images/07-polly-panel.png" alt="Polly assistant panel open in the LangSmith UI" style="width: auto; max-height: 380px; border-radius: 8px;">


### Accessing Polly

- **Keyboard shortcut:** `Cmd+I` (Mac) or `Ctrl+I` (Linux / Windows) — toggles the panel from any page.
- **Clear thread:** `Cmd+Shift+O` / `Ctrl+Shift+O` — starts a fresh conversation.
- **Click** the Polly icon in the bottom-right corner.

**Setup:** Polly needs a model provider API key configured as a workspace secret in **Settings**. Supported providers include Anthropic, OpenAI, Google Gemini, AWS Bedrock, Groq, Mistral, xAI, DeepSeek, Fireworks. If your organization restricts outbound traffic by IP, allowlist the LangSmith inference egress (your workspace admin will have the addresses).


### Questions worth asking against the client research project

After the Studio testing in Module 6, try these directly in Polly:

- *"Show me all root runs from the last hour that took more than 10 seconds."* — same as the filter DSL in Module 2, but no expression to write.
- *"Which runs failed the correctness eval? Summarize why."* — Polly reads the judge's comment field and clusters the failures.
- *"Did any users get an unknown-client response?"* — surfaces the `globex-corp` query and any similar lookups.
- *"What's the latency distribution for runs that delegated to the research subagent?"* — answers questions that would otherwise require export + analysis.
- *"Compare this experiment to the previous one. What changed?"* — when you're staring at the Experiments tab from Module 5.

Polly can also edit prompts conversationally (e.g., *"make the judge prompt stricter on citation requirements"*) — useful in the Prompt Hub. The conversational surface stays consistent; what changes is the page you're on, which scopes what Polly sees.


---

## Insights Agent

Insights is an automated analysis job that runs against a project's trace corpus and groups the runs into a hierarchy of categories — top-level themes (e.g., "requests about specific clients", "market research", "unknown identifiers") with nested subcategories. The output is an executive summary, distribution bars showing how many runs fell into each category, and per-category aggregates (latency, error rate, cost, evaluator feedback).

This is the right tool for the question "what is actually happening in production?" Patterns nobody thought to look for surface as their own categories. Insights doesn't replace the targeted queries Polly handles — it complements them by telling you which targeted queries are worth running.


### Prerequisites

- **A configured model** in the workspace settings (the analysis itself runs an LLM over the traces).
- **A healthy trace corpus.** The Studio testing in Module 6 §8 was designed to seed this — running through the twelve-query checklist twice gives Insights enough variety to find patterns. Production deployments accrue traces continuously and don't need a manual seed.
- **Plan tier:** Insights is available on LangSmith Plus and Enterprise. Confirm with your workspace admin if unsure.


### Configuring a report

1. Open the **Tracing Projects** page and select the project (`langsmith-poc` if you kept the default).
2. Click **+New → New Insights Report**.
3. Choose **auto-generation** (the agent asks guided questions about what to analyze and proposes a category structure) or **manual configuration** (specify categories, attributes, and filters yourself). Auto is the right default for a first POC report; manual is for narrow follow-ups.
4. Set the time window and any trace filter (e.g., `is_root=true` to skip child spans, or a feedback-score filter to focus on failures).
5. Launch. The job runs in the background and typically completes within 30 minutes.

<img src="../images/07-insights-config.png" alt="Insights Report configuration dialog" style="width: auto; max-height: 380px; border-radius: 8px;">


### Reading the report

The report opens to an executive summary — the largest patterns by run share, with one-sentence descriptions. From there:

- **Click a category** to drill into its subcategories and individual sample traces.
- **Distribution bars** show how many runs fell into each category, sorted by frequency.
- **Per-category aggregates** show error rate, latency percentiles, cost, and any attached evaluator feedback. A category with high frequency *and* high error rate is the first place to look for a regression.
- **Sample traces** are linked directly — clicking opens the trace in the standard trace tree view.

<img src="../images/07-insights-report.png" alt="Insights report with hierarchical categories and distribution bars" style="width: auto; max-height: 420px; border-radius: 8px;">


### Scheduling

Reports can be set to re-run on a schedule — daily, weekly on Mondays, or a custom cron expression. For a POC, run on demand against the Studio-generated traces. In production, a daily cadence is common so drift is caught the morning after it happens.

Scheduled reports notify on completion; configure notification destinations in workspace settings.


### Acting on Insights findings

Insights surfaces patterns; the loop closes via the earlier modules. When a category looks actionable:

- **High-frequency failure category** → promote sample traces to an annotation queue (Module 4) for human labeling. The labeled examples become eval data for the next iteration.
- **Latency outlier category** → inspect the trace tree to find which span is slow; tune the offending tool or prompt.
- **Missing-citation pattern** → tighten the judge prompt (Module 3) to score citation presence more strictly.
- **Drift between report runs** → diff today's executive summary against last week's. Categories that grew or shrunk are where behavior changed.


---

## Engine

> ### ⚠️ Cloud-only (Beta)
>
> Engine requires LangChain-managed inference and **does not currently support BYOK (Bring Your Own Key)**. It is **not available on self-hosted LangSmith** as of this writing. The section below is informational — review for context, then return to Polly and Insights as the failure-mode discovery surface for self-hosted deployments.


### What Engine does

Engine closes the loop that Insights opens: trace → recurring failure detected → root cause diagnosed → fix proposed → evaluator deployed. It runs continuously against the project's traces and surfaces issues without requiring manual review.

Where Insights gives you a categorized view of what's happening, Engine produces specific, actionable issues with proposed fixes — code or prompt modifications you can apply via a pull request, plus suggested evaluators to catch regressions on the same failure mode in future iterations. Insights surfaces *what's happening*; Engine surfaces *what needs to change*.


### Enabling Engine

Two configuration steps:

1. **Organization Admin** enables Engine workspace-wide in org settings.
2. **Per project**, configure:
   - Optionally connect a **GitHub repository** so fix proposals can become pull requests directly.
   - Select **priority categories** — which failure types matter most.
   - Review the **auto-generated agent overview** document Engine produces; correct any misunderstandings before issue generation starts.

Once enabled, Engine populates an **Issues** list in the project as it identifies failure patterns.


### The closed loop

Each Engine issue includes four components:

- **Diagnosis** — the root cause, linked to the failing traces. Inspect the linked traces directly to confirm the diagnosis matches your read.
- **Proposed Fix** — code or prompt changes. If a GitHub repo is connected, openable as a PR with one click.
- **Suggested Evaluator** — a check you can deploy as a new run rule (Module 3) to catch regressions on this failure mode going forward.
- **Add offline examples** — extract the failing runs as dataset entries for the experiment loop from Module 5.

<img src="../images/07-engine-issue.png" alt="Engine issue detail with diagnosis, proposed fix, and suggested evaluator" style="width: auto; max-height: 420px; border-radius: 8px;">

Priority adjustments and feedback (accept / reject / refine) train Engine's understanding of which issues matter for this specific project — relevance improves over iterations.


---

## Putting it together

A practical workflow for an enterprise POC:

1. **Generate trace volume.** Studio testing (Module 6 §8), scripted batch runs, or real production traffic. Insights and Engine both need enough traces to find patterns.
2. **Use Polly during development and review** for ad-hoc questions — slowest runs this morning, what failed correctness, what changed between experiments. Low setup, immediate.
3. **Configure an Insights report** to run on a schedule (daily is common). Review the executive summary in the morning to catch behavioral drift.
4. **When an Insights category looks actionable** — high-frequency failures, latency outliers, missed citations — promote affected traces to an annotation queue (Module 4) or a dataset (Module 5). The labeled / curated examples drive the next iteration of the judge, the agent, or both.
5. **On cloud, Engine short-circuits steps 3–4** for recurring failures: it surfaces the issue with a proposed fix and a suggested evaluator. Self-hosted readers stay on the Polly + Insights → AQ / dataset path.


## Recap

| Tool | When to use | Self-hosted |
|---|---|---|
| **Polly** | Ad-hoc questions during development or review. Lowest setup. | Confirm with workspace admin. |
| **Insights Agent** | Automated periodic discovery of patterns across a trace corpus. | Yes (Plus / Enterprise plans). |
| **Engine** | Closed-loop detection + fix proposal. | **No (cloud-only Beta).** |

All three convert raw trace data into signal worth acting on. That's the separator from an incumbent observability platform that stops at "here are the traces" — LangSmith's failure-mode discovery layer is built specifically for the scale agent applications now operate at.

**End of the module sequence.** Modules 02–07 form a continuous loop: traces in → online scoring → human-in-the-loop on flagged runs → offline experiments → deployment → automated pattern surfacing → back to dataset and judge iteration.
